# English to Hindi Machine Translation

This notebook contains the complete, self-contained implementation of an Encoder-Decoder model with Bahdanau Attention for translating English to Hindi using the `HindiEnglish Corpora` dataset on Kaggle.

> **Dataset Instructions:** Make sure you have added the `uselessnoob/english-to-hindi-machine-translation` dataset (or specifically `HindiEnglish Corpora`) to your Kaggle environment via the "Add Data" sidebar before running.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import Counter
import pandas as pd
import random
import numpy as np
import time
import os
from tqdm.notebook import tqdm

# Set seeds for reproducibility
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Core Model Architectures (`model_utils.py`)

In [ ]:
class Vocab:
    def __init__(self, counter, specials=['<unk>', '<pad>', '<sos>', '<eos>']):
        self.itos = specials + [token for token, freq in counter.items() if token not in specials]
        self.stoi = {token: i for i, token in enumerate(self.itos)}
        self.unk_index = self.stoi.get('<unk>', 0)
        self.pad_index = self.stoi.get('<pad>', 1)
        
    def __getitem__(self, token):
        return self.stoi.get(token, self.unk_index)
        
    def __len__(self):
        return len(self.itos)
    
    def __contains__(self, token):
        return token in self.stoi
        
    def lookup_token(self, index):
        return self.itos[index]

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, enc_hid_dim, dec_hid_dim, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, enc_hid_dim, bidirectional=True)
        self.fc = nn.Linear(enc_hid_dim * 2, dec_hid_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, hidden = self.rnn(embedded)
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)))
        return outputs, hidden

class Attention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim):
        super().__init__()
        self.attn = nn.Linear((enc_hid_dim * 2) + dec_hid_dim, dec_hid_dim)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)
        
    def forward(self, hidden, encoder_outputs):
        batch_size = encoder_outputs.shape[1]
        src_len = encoder_outputs.shape[0]
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        return F.softmax(attention, dim=1)

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, enc_hid_dim, dec_hid_dim, dropout, attention):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU((enc_hid_dim * 2) + emb_dim, dec_hid_dim)
        self.fc_out = nn.Linear((enc_hid_dim * 2) + dec_hid_dim + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden, encoder_outputs):
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        a = self.attention(hidden, encoder_outputs)
        a = a.unsqueeze(1)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        weighted = torch.bmm(a, encoder_outputs)
        weighted = weighted.permute(1, 0, 2)
        rnn_input = torch.cat((embedded, weighted), dim=2)
        output, hidden = self.rnn(rnn_input, hidden.unsqueeze(0))
        embedded = embedded.squeeze(0)
        output = output.squeeze(0)
        weighted = weighted.squeeze(0)
        prediction = self.fc_out(torch.cat((output, weighted, embedded), dim=1))
        return prediction, hidden.squeeze(0)

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        encoder_outputs, hidden = self.encoder(src)
        input = trg[0,:]
        for t in range(1, trg_len):
            output, hidden = self.decoder(input, hidden, encoder_outputs)
            outputs[t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1) 
            input = trg[t] if teacher_force else top1
        return outputs

## 2. Load Kaggle Dataset & Preprocess

In [ ]:
def tokenize_en(text):
    return [tok.lower() for tok in str(text).split()]

def tokenize_hi(text):
    return [tok for tok in str(text).split()]

DATASET_LIMIT = 50000 # Increase to train on more data, limit to avoid out of memory

print("Loading dataset from Kaggle inputs...")
try:
    df = pd.read_csv('/kaggle/input/hindienglish-corpora/Hindi_English_Truncated_corpus.csv')
    # Standardize column naming just in case
    if 'english_sentence' not in df.columns:
        # If there are different names, grab the first two textual columns
        cols = df.columns
        df = df.rename(columns={cols[-2]: 'english_sentence', cols[-1]: 'hindi_sentence'})
        
    # Drop NaNs
    df = df.dropna(subset=['english_sentence', 'hindi_sentence'])
    # Shuffle and limit dataset for speed processing
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    df = df.head(DATASET_LIMIT)
    
    dataset = [{'translation': {'en': row['english_sentence'], 'hi': row['hindi_sentence']}} for _, row in df.iterrows()]
except Exception as e:
    print(f"Warning: Dataset not found or error loading ({e}).\nMake sure you added the dataset to your Kaggle environment!")
    # Small dummy fallback for testing local logic if data is missing
    dataset = [{'translation': {'en': 'hello world', 'hi': 'नमस्ते दुनिया'}} for _ in range(100)]

def build_vocabs(dataset):
    print("Building English Vocabulary...")
    en_counter = Counter()
    for item in dataset:
        en_counter.update(tokenize_en(item['translation']['en']))
    en_vocab = Vocab(en_counter)
    
    print("Building Hindi Vocabulary...")
    hi_counter = Counter()
    for item in dataset:
        hi_counter.update(tokenize_hi(item['translation']['hi']))
    hi_vocab = Vocab(hi_counter)
    
    return en_vocab, hi_vocab

en_vocab, hi_vocab = build_vocabs(dataset)
print(f"Unique tokens in En (source): {len(en_vocab)}")
print(f"Unique tokens in Hi (target): {len(hi_vocab)}")

def preprocess_data(dataset, en_vocab, hi_vocab):
    processed_data = []
    for item in dataset:
        src_tokens = tokenize_en(item['translation']['en'])
        trg_tokens = tokenize_hi(item['translation']['hi'])
        
        # Skip too long sentences to keep memory low
        if len(src_tokens) > 50 or len(trg_tokens) > 50:
            continue
            
        src = [en_vocab['<sos>']] + [en_vocab[token] for token in src_tokens] + [en_vocab['<eos>']]
        trg = [hi_vocab['<sos>']] + [hi_vocab[token] for token in trg_tokens] + [hi_vocab['<eos>']]
        processed_data.append((torch.tensor(src), torch.tensor(trg)))
    return processed_data

train_data = preprocess_data(dataset, en_vocab, hi_vocab)
print(f"Total processed training pairs: {len(train_data)}")

def collate_fn(batch):
    src_list, trg_list = [], []
    for src, trg in batch:
        src_list.append(src)
        trg_list.append(trg)
    src_list = torch.nn.utils.rnn.pad_sequence(src_list, padding_value=en_vocab['<pad>'])
    trg_list = torch.nn.utils.rnn.pad_sequence(trg_list, padding_value=hi_vocab['<pad>'])
    return src_list, trg_list

train_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True, collate_fn=collate_fn)

## 3. Initialize Model

In [ ]:
INPUT_DIM = len(en_vocab)
OUTPUT_DIM = len(hi_vocab)
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
ENC_HID_DIM = 512
DEC_HID_DIM = 512
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5

attn = Attention(ENC_HID_DIM, DEC_HID_DIM)
enc = Encoder(INPUT_DIM, ENC_EMB_DIM, ENC_HID_DIM, DEC_HID_DIM, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, ENC_HID_DIM, DEC_HID_DIM, DEC_DROPOUT, attn)

model = Seq2Seq(enc, dec, device).to(device)

def init_weights(m):
    for name, param in m.named_parameters():
        if 'weight' in name:
            nn.init.normal_(param.data, mean=0, std=0.01)
        else:
            nn.init.constant_(param.data, 0)
            
model.apply(init_weights)

optimizer = optim.Adam(model.parameters())
TRG_PAD_IDX = hi_vocab['<pad>']
criterion = nn.CrossEntropyLoss(ignore_index=TRG_PAD_IDX)

## 4. Training Loop

In [ ]:
def train(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    # Wrap the iterator with tqdm for a progress bar
    for i, (src, trg) in enumerate(tqdm(iterator, leave=False, desc="Batches")):
        src = src.to(device)
        trg = trg.to(device)
        
        optimizer.zero_grad()
        output = model(src, trg)
        
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        trg = trg[1:].view(-1)
        
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
        
    return epoch_loss / len(iterator)

N_EPOCHS = 10
CLIP = 1

print("Starting training...")
for epoch in range(N_EPOCHS):
    start_time = time.time()
    train_loss = train(model, train_loader, optimizer, criterion, CLIP)
    end_time = time.time()
    
    print(f'Epoch: {epoch+1:02} | Time: {end_time - start_time:.2f}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {np.exp(train_loss):7.3f}')

## 5. Export Model & Vocabulary to `/kaggle/working/`

In [ ]:
OUTPUT_MODEL_PATH = '/kaggle/working/en_hi_model.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'en_vocab': en_vocab,
    'hi_vocab': hi_vocab,
    'params': {
        'input_dim': INPUT_DIM,
        'output_dim': OUTPUT_DIM,
        'enc_emb_dim': ENC_EMB_DIM,
        'dec_emb_dim': DEC_EMB_DIM,
        'enc_hid_dim': ENC_HID_DIM,
        'dec_hid_dim': DEC_HID_DIM,
        'enc_dropout': ENC_DROPOUT,
        'dec_dropout': DEC_DROPOUT
    }
}, OUTPUT_MODEL_PATH)

print(f"Model completely processed and saved to: {OUTPUT_MODEL_PATH}")
print("You can now download 'en_hi_model.pt' from the right sidebar 'Output' panel and use it in your Streamlit application!")